In [5]:
import gymnasium as gym
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

Share Scaffolding

Multi-Layer Perceptron

In [ ]:
def MLP(input_dim: int, output_dim: int, hidden_dims=[64, 64], activation=torch.nn.ReLU):
    layers = []
    prev_dim = input_dim
    for hidden_dim in hidden_dims:
        layers.append(torch.nn.Linear(prev_dim, hidden_dim))
        layers.append(activation())
        prev_dim = hidden_dim
    layers.append(torch.nn.Linear(prev_dim, output_dim))
    return torch.nn.Sequential(*layers)

Evaluate the Policy

In [6]:
def evaluate(policy: nn.Module, env: gym.Env, n_episodes: int):
    with torch.no_grad():
        policy.eval()
        episode_returns = []
        for episode in range(n_episodes):
            obs, info = env.reset()
            done = False
            truncated = False
            episode_return = 0.0

            while not (done or truncated):
                obs_tensor = torch.FloatTensor(obs).unsqueeze(0)
                action_logits = policy(obs_tensor)
                action = torch.argmax(action_logits, dim=-1).item()
                obs, reward, terminated, truncated, info = env.step(action)
                episode_return += float(reward)
            episode_returns.append(episode_return)

    mean_return = np.mean(episode_returns)
    return mean_return

make env

In [7]:
def make_env(env_id: str, seed: int):
    env = gym.make(env_id)
    env.action_space.seed(seed)
    env.observation_space.seed(seed)
    torch.manual_seed(seed)
    np.random.default_rng(seed)
    return env